# 01 — 数据总览(xc 优先,任意产品可用)

改开头 `PRODUCT` 切换产品。xc 系产品(`xc_resp_finish` / `xc_qual_finish` / `xc_qual_finish_1v1` / `xc_e2e_credit` / `xc_e2e_credit_1v1`)的先决条件:已跑 `scripts/build_xc_dataset.py` 生成 `data/xc_full.csv` 与 `data/xc_qual_finish.csv`。

覆盖 runbook 第 1 步检查点:标签/档位分布、全流程分析漏斗(对照业务大盘,量级偏差 >20% 即停)、标签成熟度(尾部月份授信率走低 = 授信结果未回流)、join 质检、缺失概览。非 xc 产品(如 `bank_marketing`)运行时,xc 专属格自动跳过。

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from wdm.config import load_config
from wdm.io.column_scanner import scan_columns

PRODUCT = 'xc_resp_finish'   # 任意 configs/products/*.yaml 的 name

cfg = load_config(PRODUCT)
idx = scan_columns(cfg)
label_col = idx['label_column']
time_col = cfg['data'].get('time_column')
print('product:', PRODUCT)
print('train_path:', cfg['data']['train_path'])
print('label:', label_col, '| time:', time_col)
print('id/outcome columns(不入特征):', cfg['data'].get('id_columns'))
print('features:', len(idx['features']))
print('dtype 分布:', pd.Series(idx['dtypes']).value_counts().to_dict())

In [ ]:
data_path = Path(cfg['_repo_root']) / cfg['data']['train_path']
df = pd.read_csv(data_path)
print('rows:', len(df), '| cols:', df.shape[1])
print('label {0} positive rate: {1:.4f}'.format(label_col, df[label_col].mean()))
if time_col and time_col in df.columns:
    dtv = pd.to_numeric(df[time_col], errors='coerce')
    print('dt 范围: {0} .. {1} | dt 为 NaN 的行: {2}'.format(
        int(dtv.min()), int(dtv.max()), int(dtv.isna().sum())))
# 千维宽表只预览 meta + 前几个特征列,避免刷屏
meta_cols = [c for c in (cfg['data'].get('id_columns') or []) + [time_col, label_col]
             if c and c in df.columns]
preview = meta_cols + [c for c in idx['features'] if c in df.columns][:8]
df[preview].head()

In [ ]:
# xc 结果列一览:各漏斗 flag 正例率 + 原始 credit_1v1 档位分布(非 xc 自动跳过)
OUTCOME_COLS = ['is_reg', 'is_finish_task', 'is_credit_succ', 'is_credit_1v1']
present = [c for c in OUTCOME_COLS if c in df.columns]
if present:
    display(pd.DataFrame({'n_pos': df[present].sum().astype(int),
                          'positive_rate': df[present].mean().round(4)}))
else:
    print('无 xc 漏斗结果列,跳过')
if 'credit_1v1' in df.columns:
    tiers = pd.to_numeric(df['credit_1v1'], errors='coerce')
    vc = tiers.value_counts(dropna=False).sort_index()
    print('credit_1v1 档位分布(1/2/3 为正样本档,业务价值 120/290/790;0、-1 为负):')
    display(pd.DataFrame({'n': vc, 'pct': (vc / len(df)).round(4)}))
    n_bad = int((tiers.notna() & ~tiers.isin([1, 2, 3, 0, -1])).sum())
    if n_bad:
        print('警告:' + str(n_bad) + ' 行 credit_1v1 在 {1,2,3,0,-1} 之外(合表时已按负样本处理)')

In [ ]:
# 全流程分析漏斗(口径与 build_xc_dataset.py 打印一致,基于全量表 xc_full.csv)。
# 检查点:逐段转化须与业务大盘一致(量级偏差 >20% 即停,排查 join 匹配率与标签口径)
FUNNELS = [['is_reg', 'is_finish_task', 'is_credit_succ'],
           ['is_reg', 'is_finish_task', 'is_credit_1v1']]
full_path = Path(cfg['_repo_root']) / 'data/xc_full.csv'
if full_path.is_file():
    funnel_cols = sorted(set(c for f in FUNNELS for c in f))
    base = (df[funnel_cols] if cfg['data']['train_path'].endswith('xc_full.csv')
            else pd.read_csv(full_path, usecols=funnel_cols))
    n_total = len(base)
    rows = []
    for funnel in FUNNELS:
        pop = base
        for stage in funnel:
            n_prev = len(pop)
            pop = pop[pop[stage] == 1]
            rows.append({'endpoint': funnel[-1], 'stage': stage, 'n': len(pop),
                         'step_cvr': round(len(pop) / n_prev, 4) if n_prev else 0.0,
                         'cum_cvr': round(len(pop) / n_total, 4) if n_total else 0.0})
    display(pd.DataFrame(rows))
else:
    print('无 data/xc_full.csv(非 xc 产品或未合表),跳过分析漏斗')

In [ ]:
# 标签成熟度:按月看样本量与各结果列正例率。尾部月份授信率系统性走低 → 标签未成熟
# (授信结果未回流),该窗口不应进训练/评估(runbook 第 0 步前置条件)
if time_col and time_col in df.columns and present:
    dtv = pd.to_numeric(df[time_col], errors='coerce')
    ok = dtv.notna()
    month = (dtv[ok] // 100).astype(int)
    cnt = month.value_counts().sort_index()
    rates = df.loc[ok, present].groupby(month).mean()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    cnt.plot(kind='bar', ax=axes[0], title='rows per month')
    rates.plot(ax=axes[1], marker='o', title='positive rate per month')
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    display(pd.DataFrame({'n': cnt}).join(rates.round(4)))
else:
    print('无时间列或无 xc 结果列,跳过')

In [ ]:
# 合表质检:重复 (id, dt) 键;qual 表行数应等于 full 表中 is_finish_task==1 的行数
if 'id' in df.columns and time_col and time_col in df.columns:
    dup = int(df.duplicated(subset=['id', time_col]).sum())
    print('重复 (id, dt) 行:', dup, '' if dup == 0 else '<- 回查 build_xc_dataset 的重复键告警')
qual_path = Path(cfg['_repo_root']) / 'data/xc_qual_finish.csv'
if full_path.is_file() and qual_path.is_file():
    n_finish = int((pd.read_csv(full_path, usecols=['is_finish_task'])['is_finish_task'] == 1).sum())
    n_qual = len(pd.read_csv(qual_path, usecols=['is_finish_task']))
    status = 'OK' if n_finish == n_qual else '不一致!重跑 build_xc_dataset.py'
    print('full 表 is_finish_task==1: {0} | qual 表行数: {1} -> {2}'.format(n_finish, n_qual, status))
else:
    print('xc 两张表不全,跳过 full/qual 一致性检查')

In [ ]:
# 特征缺失概览(xc 走 keep_nan 不填充;此处看原始 NaN/空口径。
# 含 0/负哨兵的精确缺失口径以 Stage-1 报告 report/missing.csv 为准)
feat_cols = [c for c in idx['features'] if c in df.columns]
miss = df[feat_cols].isna().mean().sort_values(ascending=False)
print(miss.describe().round(4))
ax = miss.plot(kind='hist', bins=40, figsize=(7, 3.5), title='per-feature missing rate')
ax.set_xlabel('missing rate')
mr_max = cfg['analysis'].get('missing_rate_max')
if mr_max is not None:
    print('缺失率超过 missing_rate_max={0} 的特征数: {1}(Stage-1 会硬过滤)'.format(
        mr_max, int((miss > mr_max).sum())))
print('缺失率最高的 15 个特征:')
display(miss.head(15).round(4))